# 03 — Beitragsextraktion (SHAP / EBM)

Extrahiert globale Feature-Importance und lokale Beiträge für die **5 fixen Testinstanzen**
(`INSTANCE_IDS = [42, 100, 250, 500, 1337]`) für XGBoost und EBM mit **Option 2 (Poisson-Log)**.

- **XGBoost**: SHAP TreeExplainer — Beiträge im Log-Raum (additiv)
- **EBM**: interpret `explain_local` — Terme ebenfalls im Log-Raum

Alle Ausgaben werden als JSON in `explanations/` gespeichert und von Notebooks 04–06 gelesen.

In [1]:
from __future__ import annotations

import sys, json
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd

from utils import (
    INSTANCE_IDS, scale_instance_ids,
    EXPLANATIONS_DIR, MODELS_DIR, RESULTS_DIR,
)
from utils.data import load_train_test
from utils.explanations import build_global, build_local, save_explanation

LOSS_KEY = "poisson_log"

# Phase 3b: lokale Erklärungen werden gen-unabhängig je Instanz gebraucht. Damit
# sowohl die n=20-Validität (INSTANCE_IDS) als auch der n≈200-Skalierungslauf
# (scale_instance_ids) abgedeckt sind, über die VEREINIGUNG generieren. Das
# Re-Erzeugen der 10 Validitäts-Instanzen ist deterministisch (kein API-Call).
ALL_INSTANCE_IDS = sorted(set(INSTANCE_IDS) | set(scale_instance_ids()))

print(f"Validitäts-Instanzen: {len(INSTANCE_IDS)}")
print(f"Gesamt (inkl. Skalierung): {len(ALL_INSTANCE_IDS)} Instanzen")
print(f"Loss-Option:   {LOSS_KEY}")
print(f"Ausgabe:       {EXPLANATIONS_DIR}")

Validitäts-Instanzen: 10
Gesamt (inkl. Skalierung): 15 Instanzen
Loss-Option:   poisson_log
Ausgabe:       /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/explanations


## 1 · Daten und Modelle laden

In [2]:
X_train, y_train, X_test, y_test = load_train_test()
print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")

from utils.models import load_models
xgb, ebm = load_models(LOSS_KEY)
print("Modelle geladen.")

metrics = json.loads((RESULTS_DIR / f"model_metrics_{LOSS_KEY}.json").read_text())["metrics"]
print("Metriken geladen:", list(metrics.keys()))

X_train: (12152, 9)  |  X_test: (5227, 9)
Modelle geladen.
Metriken geladen: ['xgb', 'ebm']


## 2 · Globale Erklärungen

In [3]:
print("Berechne globale XGBoost-Erklärung (SHAP über Trainingsset)...")
global_xgb = build_global(xgb, "xgb", X_train, metrics["xgb"])

path = save_explanation(global_xgb, f"global_xgb_{LOSS_KEY}.json")
print(f"Gespeichert: {path}")

print("\nTop-5 Features (XGBoost):")
for entry in global_xgb["global_importance"][:5]:
    print(f"  #{entry['rank']:2d}  {entry['feature']:12s}  {entry['importance']:.4f}")

Berechne globale XGBoost-Erklärung (SHAP über Trainingsset)...
Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/explanations/global_xgb_poisson_log.json

Top-5 Features (XGBoost):
  # 1  hr            0.8326
  # 2  yr            0.2004
  # 3  temp          0.1971
  # 4  weekday       0.1728
  # 5  mnth          0.1183


In [4]:
print("Berechne globale EBM-Erklärung...")
global_ebm = build_global(ebm, "ebm", X_train, metrics["ebm"])

path = save_explanation(global_ebm, f"global_ebm_{LOSS_KEY}.json")
print(f"Gespeichert: {path}")

print("\nTop-5 Features (EBM):")
for entry in global_ebm["global_importance"][:5]:
    print(f"  #{entry['rank']:2d}  {entry['feature']:12s}  {entry['importance']:.4f}")

Berechne globale EBM-Erklärung...
Gespeichert: /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/explanations/global_ebm_poisson_log.json

Top-5 Features (EBM):
  # 1  hr            0.8888
  # 2  temp          0.2526
  # 3  yr            0.2166
  # 4  mnth          0.0857
  # 5  hum           0.0826


## 3 · Lokale Erklärungen für die 5 Testinstanzen

In [5]:
for n_done, iid in enumerate(ALL_INSTANCE_IDS, 1):
    local = build_local(xgb, "xgb", X_test, y_test, iid)
    path = save_explanation(local, f"local_xgb_{LOSS_KEY}_inst{iid}.json")
    if n_done <= 5 or n_done % 25 == 0 or iid in INSTANCE_IDS:
        top = local["contributions"][0]
        print(f"  [{n_done:3d}/{len(ALL_INSTANCE_IDS)}] inst={iid:4d}  pred={local['prediction']:6.1f}  "
              f"y_true={local['y_true']:6.0f}  top={top['feature']} ({top['contribution']:+.3f})")
print(f"XGB: {len(ALL_INSTANCE_IDS)} lokale Erklärungen gespeichert.")

  [  1/15] inst= 224  pred= 112.0  y_true=    96  top=temp (-0.549)
  [  2/15] inst= 419  pred=   2.1  y_true=     4  top=hr (-3.076)
  [  3/15] inst= 580  pred=  64.6  y_true=    63  top=hr (-0.365)
  [  4/15] inst=1041  pred= 385.4  y_true=   387  top=hr (+0.746)
  [  5/15] inst=1481  pred= 228.6  y_true=   277  top=hr (+0.378)
  [  6/15] inst=1677  pred= 254.3  y_true=   286  top=hr (+0.252)
  [  7/15] inst=2058  pred= 194.1  y_true=   243  top=yr (-0.195)
  [  8/15] inst=2510  pred= 326.0  y_true=   372  top=hr (+0.686)
  [  9/15] inst=3543  pred= 309.9  y_true=   286  top=yr (+0.186)
  [ 14/15] inst=3847  pred= 585.8  y_true=   531  top=hr (+0.413)
  [ 15/15] inst=4454  pred= 297.3  y_true=   354  top=hr (+0.352)
XGB: 15 lokale Erklärungen gespeichert.


In [6]:
for n_done, iid in enumerate(ALL_INSTANCE_IDS, 1):
    local = build_local(ebm, "ebm", X_test, y_test, iid)
    path = save_explanation(local, f"local_ebm_{LOSS_KEY}_inst{iid}.json")
    if n_done <= 5 or n_done % 25 == 0 or iid in INSTANCE_IDS:
        top = local["contributions"][0]
        print(f"  [{n_done:3d}/{len(ALL_INSTANCE_IDS)}] inst={iid:4d}  pred={local['prediction']:6.1f}  "
              f"y_true={local['y_true']:6.0f}  top={top['feature']} ({top['contribution']:+.3f})")
print(f"EBM: {len(ALL_INSTANCE_IDS)} lokale Erklärungen gespeichert.")

  [  1/15] inst= 224  pred= 117.0  y_true=    96  top=hr (+0.852)
  [  2/15] inst= 419  pred=   2.3  y_true=     4  top=hr (-2.183)
  [  3/15] inst= 580  pred=  73.1  y_true=    63  top=yr (-0.226)
  [  4/15] inst=1041  pred= 389.7  y_true=   387  top=hr (+1.109)
  [  5/15] inst=1481  pred= 241.8  y_true=   277  top=hr (+0.819)
  [  6/15] inst=1677  pred= 260.2  y_true=   286  top=hr (+0.555)
  [  7/15] inst=2058  pred= 176.9  y_true=   243  top=hr (+0.555)
  [  8/15] inst=2510  pred= 311.4  y_true=   372  top=hr (+1.148)
  [  9/15] inst=3543  pred= 296.0  y_true=   286  top=hr (+0.555)
  [ 14/15] inst=3847  pred= 593.4  y_true=   531  top=hr (+0.661)
  [ 15/15] inst=4454  pred= 331.8  y_true=   354  top=hr (+0.852)
EBM: 15 lokale Erklärungen gespeichert.


## 4 · Überblick gespeicherter Dateien

In [7]:
files = sorted(EXPLANATIONS_DIR.glob(f"*_{LOSS_KEY}*.json"))
print(f"Gespeicherte Erklärungsdateien ({len(files)}):")
for f in files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45s}  {size_kb:.1f} KB")

Gespeicherte Erklärungsdateien (32):
  global_ebm_poisson_log.json                    2.8 KB
  global_xgb_poisson_log.json                    2.7 KB
  local_ebm_poisson_log_inst1041.json            1.2 KB
  local_ebm_poisson_log_inst1481.json            1.2 KB
  local_ebm_poisson_log_inst1677.json            1.2 KB
  local_ebm_poisson_log_inst2058.json            1.2 KB
  local_ebm_poisson_log_inst224.json             1.1 KB
  local_ebm_poisson_log_inst2510.json            1.2 KB
  local_ebm_poisson_log_inst3543.json            1.2 KB
  local_ebm_poisson_log_inst3608.json            1.2 KB
  local_ebm_poisson_log_inst3658.json            1.2 KB
  local_ebm_poisson_log_inst3687.json            1.2 KB
  local_ebm_poisson_log_inst3725.json            1.2 KB
  local_ebm_poisson_log_inst3847.json            1.2 KB
  local_ebm_poisson_log_inst419.json             1.2 KB
  local_ebm_poisson_log_inst4454.json            1.2 KB
  local_ebm_poisson_log_inst580.json             1.2 KB
  local_xgb

## 5 · Sanity-Check: XGBoost vs. EBM — Top-Features im Vergleich

In [8]:
imp_xgb = {e["feature"]: e["rank"] for e in global_xgb["global_importance"]}
imp_ebm = {e["feature"]: e["rank"] for e in global_ebm["global_importance"]}

all_features = sorted(set(imp_xgb) | set(imp_ebm))
rows = [{"Feature": f, "Rank XGB": imp_xgb.get(f, "-"), "Rank EBM": imp_ebm.get(f, "-")} for f in all_features]
df = pd.DataFrame(rows).sort_values("Rank XGB")
print(df.to_string(index=False))

   Feature  Rank XGB  Rank EBM
        hr         1         1
        yr         2         3
      temp         3         2
   weekday         4         7
      mnth         5         4
       hum         6         5
weathersit         7         6
 windspeed         8         8
   holiday         9         9
